# 07 - Predictive Analysis (Classification)
Logistic Regression, SVM, Decision Tree and KNN tuned with GridSearchCV.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
data=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_2.csv')
X=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_3.csv')
Y=data['Class'].to_numpy()
X=preprocessing.StandardScaler().fit(X).transform(X)
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2)
print(X_test.shape)

In [ ]:
def plot_cm(y,yhat,title):
    cm=confusion_matrix(y,yhat);ax=plt.subplot();sns.heatmap(cm,annot=True,ax=ax,fmt='d')
    ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.set_title(title)
    ax.xaxis.set_ticklabels(['did not land','landed']);ax.yaxis.set_ticklabels(['did not land','landed']);plt.show()
models={
 'Logistic Regression':(LogisticRegression(),{'C':[0.01,0.1,1],'penalty':['l2'],'solver':['lbfgs']}),
 'SVM':(SVC(),{'kernel':['linear','rbf','poly','sigmoid'],'C':np.logspace(-3,3,5),'gamma':np.logspace(-3,3,5)}),
 'Decision Tree':(DecisionTreeClassifier(),{'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':[2*n for n in range(1,10)],'max_features':['sqrt','log2'],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}),
 'KNN':(KNeighborsClassifier(),{'n_neighbors':list(range(1,11)),'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]})}
results={}
for name,(est,grid) in models.items():
    gs=GridSearchCV(est,grid,cv=10).fit(X_train,Y_train)
    results[name]=(gs.best_score_,gs.score(X_test,Y_test))
    print(name,gs.best_params_,'CV acc:',round(gs.best_score_,4),'Test acc:',round(gs.score(X_test,Y_test),4))
    plot_cm(Y_test,gs.predict(X_test),name)

In [ ]:
res=pd.DataFrame(results,index=['CV accuracy','Test accuracy']).T
res.plot(kind='bar',ylim=(0.5,1));plt.title('Model comparison');plt.show()
print('Best model by CV:',res['CV accuracy'].idxmax())